In [ ]:
%pip install pymupdf spacy rank-bm25 faiss-cpu

In [ ]:
%pip install sentence-transformers transformers accelerate streamlit rouge-score bert-score datasets pytest

In [6]:
%pip install import-ipynb

  Using cached nbformat-5.11.1-py3-none-any.whl.metadata (3.7 kB)
  Using cached fastjsonschema-2.22.2-py3-none-any.whl.metadata (2.1 kB)
Using cached nbformat-5.11.1-py3-none-any.whl (79 kB)
Using cached fastjsonschema-2.22.2-py3-none-any.whl (27 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [import-ipynb] [nbformat]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0)
if torch.cuda.is_available() else 'CPU')

In [2]:
import pymupdf
from pathlib import Path
import fitz


def extract_pdf_text(pdf_path):
    """Extract non-empty pages from a PDF."""
    pdf_path = Path(pdf_path)

    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

    if pdf_path.suffix.lower() != ".pdf":
        raise ValueError("Only PDF files are supported.")

    pages = []

    with fitz.open(pdf_path) as document:
        if document.is_encrypted:
            raise ValueError("Encrypted PDFs are not supported.")

        for page_number, page in enumerate(document, start=1):
            text = page.get_text("text").strip()

            if text:
                pages.append({
                    "document": pdf_path.name,
                    "page": page_number,
                    "text": text
                })

    if not pages:
        raise ValueError(
            "No selectable text found. The PDF may be scanned or image-only."
        )

    return pages


def get_pdf_info(pdf_path):
    """Return basic metadata about a PDF."""
    with fitz.open(pdf_path) as document:
        return {
            "filename": Path(pdf_path).name,
            "total_pages": len(document),
            "title": document.metadata.get("title") or Path(pdf_path).stem
        }